In [ ]:
# SPDX-License-Identifier: BUSL-1.1
# Copyright 2026 Ryan Gillespie / Optitransfer
# Licensed under the Business Source License 1.1
# https://github.com/mgillr/crdt-merge/blob/main/LICENSE
# Change Date: 2028-03-29 | Change License: Apache-2.0

# crdt-merge v0.8.2 — A100 Benchmark Suite
# Run on: Google Colab with A100 GPU Runtime
!pip install -q crdt-merge[all] torch numpy

import crdt_merge
print(f"crdt-merge v{crdt_merge.__version__}")

In [ ]:
import time, sys, platform, random, os
import numpy as np

print(f"Python: {sys.version}")
print(f"Platform: {platform.system()} {platform.machine()}")

try:
    import torch
    print(f"PyTorch: {torch.__version__}")
    print(f"CUDA: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        props = torch.cuda.get_device_properties(0)
        print(f"Memory: {props.total_mem / 1e9:.1f} GB")
except ImportError:
    print("PyTorch not installed — GPU benchmarks will be skipped")
    torch = None

# Helper functions
WIDTH = 80

def row(label, value, unit=""):
    combined = f"{value} {unit}".strip()
    print(f"  {label:<48s} {combined:>28s}")

def divider():
    print("  " + "-" * (WIDTH - 4))

def fmt_ops(ops):
    if ops >= 1e9: return f"{ops / 1e9:.2f}B"
    if ops >= 1e6: return f"{ops / 1e6:.2f}M"
    if ops >= 1e3: return f"{ops / 1e3:.1f}K"
    return f"{ops:.1f}"

def fmt_time(seconds):
    if seconds < 1e-6: return f"{seconds * 1e9:.1f} ns"
    if seconds < 1e-3: return f"{seconds * 1e6:.2f} µs"
    if seconds < 1: return f"{seconds * 1e3:.2f} ms"
    return f"{seconds:.3f} s"

# Accumulate results for summary
ALL_RESULTS = []

def record(category, test, value, unit=""):
    ALL_RESULTS.append({"category": category, "test": test, "value": value, "unit": unit})

## 🧠 Context Memory Benchmarks

Testing ContextBloom, MemorySidecar, ContextMerge, and ContextConsolidator.

In [ ]:
# SPDX-License-Identifier: BUSL-1.1
# Copyright 2026 Ryan Gillespie / Optitransfer
# Licensed under the Business Source License 1.1
# https://github.com/mgillr/crdt-merge/blob/main/LICENSE
# Change Date: 2028-03-29 | Change License: Apache-2.0

from crdt_merge.context.bloom import ContextBloom
from crdt_merge.context.sidecar import MemorySidecar
from crdt_merge.context.merge import ContextMerge
from crdt_merge.context.consolidator import ContextConsolidator
from crdt_merge.context import MemoryChunk

print("=" * WIDTH)
print("  Context Memory Benchmarks")
print("=" * WIDTH)

# ── ContextBloom ──
print("\n  ContextBloom:")
for n in [100_000, 1_000_000, 10_000_000]:
    cb = ContextBloom(expected_items=n)
    t0 = time.perf_counter()
    for i in range(n):
        cb.add(f"key_{i}")
    add_elapsed = time.perf_counter() - t0

    t0 = time.perf_counter()
    for i in range(n):
        cb.contains(f"key_{i}")
    contains_elapsed = time.perf_counter() - t0

    label = fmt_ops(n)
    add_ops = n / add_elapsed
    contains_ops = n / contains_elapsed
    row(f"  add ({label} items)", fmt_ops(add_ops), "ops/sec")
    row(f"  contains ({label} items)", fmt_ops(contains_ops), "ops/sec")
    row(f"    avg add latency", fmt_time(add_elapsed / n))
    row(f"    avg contains latency", fmt_time(contains_elapsed / n))
    record("ContextBloom", f"add {label}", fmt_ops(add_ops), "ops/sec")
    record("ContextBloom", f"contains {label}", fmt_ops(contains_ops), "ops/sec")
    divider()

# ── MemorySidecar ──
print("\n  MemorySidecar:")
n = 100_000
t0 = time.perf_counter()
sidecars = []
for i in range(n):
    ms = MemorySidecar(
        fact_id=f"fact_{i}",
        content_hash=f"hash_{i:08x}",
        topic=random.choice(["science", "history", "math", "code"]),
        source_agent=f"agent_{i % 10}",
        confidence=random.uniform(0.5, 1.0),
        tags=[random.choice(["important", "verified", "recent"])],
    )
    sidecars.append(ms)
create_elapsed = time.perf_counter() - t0
row(f"  Create {fmt_ops(n)} sidecars", fmt_time(create_elapsed))
record("MemorySidecar", "create 100K", fmt_time(create_elapsed))

t0 = time.perf_counter()
matches = sum(1 for ms in sidecars if ms.matches_filter(topic="science", min_confidence=0.7))
filter_elapsed = time.perf_counter() - t0
filter_ops = n / filter_elapsed
row(f"  Filter {fmt_ops(n)} sidecars", fmt_ops(filter_ops), "ops/sec")
row(f"    avg filter latency", fmt_time(filter_elapsed / n))
record("MemorySidecar", "filter 100K", fmt_ops(filter_ops), "ops/sec")
divider()

# ── ContextMerge ──
print("\n  ContextMerge (2 agents):")
def make_mems(agent_id, n):
    return [
        {
            "fact": f"memory_{agent_id}_{i}_content",
            "source": agent_id,
            "topic": "bench",
            "confidence": random.uniform(0.6, 1.0),
        }
        for i in range(n)
    ]

for n in [1_000, 10_000, 100_000]:
    mems_a = make_mems("a", n)
    mems_b = make_mems("b", n)
    cm = ContextMerge(strategy="lww")
    t0 = time.perf_counter()
    result = cm.merge(mems_a, mems_b)
    elapsed = time.perf_counter() - t0
    label = fmt_ops(n)
    row(f"  2 agents × {label} memories", fmt_time(elapsed))
    record("ContextMerge", f"2×{label}", fmt_time(elapsed))
    divider()

# ── ContextConsolidator ──
print("\n  ContextConsolidator:")
for n in [10_000, 50_000]:
    chunks = []
    for i in range(n):
        sidecar = MemorySidecar(
            fact_id=f"fact_{i}", content_hash=f"h_{i:08x}",
            topic=random.choice(["a", "b", "c"]), source_agent="agent_0",
        )
        chunks.append(MemoryChunk(fact=f"memory_{i}", sidecar=sidecar))
    cc = ContextConsolidator()
    t0 = time.perf_counter()
    blocks = cc.consolidate(chunks)
    elapsed = time.perf_counter() - t0
    label = fmt_ops(n)
    row(f"  {label} memories → {len(blocks)} blocks", fmt_time(elapsed))
    record("Consolidator", f"{label}→blocks", fmt_time(elapsed))
    divider()

print("\n✅ Context Memory benchmarks complete")

## 🤖 Agentic AI Benchmarks

Testing AgentState.merge and SharedKnowledge.merge.

In [ ]:
# SPDX-License-Identifier: BUSL-1.1
# Copyright 2026 Ryan Gillespie / Optitransfer
# Licensed under the Business Source License 1.1
# https://github.com/mgillr/crdt-merge/blob/main/LICENSE
# Change Date: 2028-03-29 | Change License: Apache-2.0

from crdt_merge.agentic import AgentState, SharedKnowledge

print("=" * WIDTH)
print("  Agentic AI Benchmarks")
print("=" * WIDTH)

# ── AgentState.merge ──
print("\n  AgentState.merge:")
for n_facts in [100, 500, 1000, 5000, 10000]:
    a = AgentState(agent_id="agent_a")
    b = AgentState(agent_id="agent_b")
    for i in range(n_facts):
        a.add_fact(f"fact_{i}", f"val_a_{i}", confidence=0.9)
        b.add_fact(f"fact_{i}", f"val_b_{i}", confidence=0.8)
    for i in range(n_facts // 2):
        a.add_fact(f"unique_a_{i}", f"u_a_{i}")
        b.add_fact(f"unique_b_{i}", f"u_b_{i}")

    n_iters = max(1, 1000 // n_facts)
    t0 = time.perf_counter()
    for _ in range(n_iters):
        merged = a.merge(b)
    elapsed = time.perf_counter() - t0
    avg = elapsed / n_iters
    row(f"  {n_facts} facts", fmt_time(avg), f"({fmt_ops(n_iters/elapsed)} merges/sec)")
    record("AgentState", f"merge {n_facts} facts", fmt_time(avg))
    divider()

# ── SharedKnowledge.merge ──
print("\n  SharedKnowledge.merge:")
for n_agents in [2, 5, 10, 20]:
    facts_per = 200
    agents = []
    for a_idx in range(n_agents):
        agent = AgentState(agent_id=f"agent_{a_idx}")
        for f_idx in range(facts_per):
            agent.add_fact(
                f"shared_{f_idx}",
                f"v_{a_idx}_{f_idx}",
                confidence=random.uniform(0.5, 1.0),
            )
        for f_idx in range(50):
            agent.add_fact(f"unique_{a_idx}_{f_idx}", f"u_{a_idx}_{f_idx}")
        agents.append(agent)


    n_iters = max(1, 100 // n_agents)
    t0 = time.perf_counter()
    for _ in range(n_iters):
        # Using classmethod SharedKnowledge.merge(*agents)
        result = SharedKnowledge.merge(*agents)
    elapsed = time.perf_counter() - t0
    avg = elapsed / n_iters
    row(f"  {n_agents} agents × {facts_per} facts", fmt_time(avg))
    record("SharedKnowledge", f"{n_agents} agents", fmt_time(avg))
    divider()

print("\n✅ Agentic AI benchmarks complete")

## 🔬 ModelCRDT GPU Benchmarks

Testing model merge strategies with large tensors. Requires PyTorch + CUDA.

In [ ]:
# SPDX-License-Identifier: BUSL-1.1
# Copyright 2026 Ryan Gillespie / Optitransfer
# Licensed under the Business Source License 1.1
# https://github.com/mgillr/crdt-merge/blob/main/LICENSE
# Change Date: 2028-03-29 | Change License: Apache-2.0

from crdt_merge.model.core import ModelMerge, ModelMergeSchema

if torch is None or not torch.cuda.is_available():
    print("⚠️  Skipping GPU benchmarks — no CUDA available")
else:
    device = "cuda"
    print("=" * WIDTH)
    print("  ModelCRDT GPU Benchmarks")
    print("=" * WIDTH)

    strategies = ["weight_average", "ties", "dare"]

    for strat_name in strategies:
        print(f"\n  Strategy: {strat_name}")
        schema = ModelMergeSchema(strategies={"default": strat_name})
        mm = ModelMerge(schema=schema)

        for n_params in [1_000_000, 10_000_000, 100_000_000]:
            # Create models as numpy arrays (ModelMerge handles internals)
            model_a = {"layer.weight": np.random.randn(n_params).astype(np.float32)}
            model_b = {"layer.weight": np.random.randn(n_params).astype(np.float32)}
            base_model = {"layer.weight": np.random.randn(n_params).astype(np.float32)}

            # Warm up
            try:
                _ = mm.merge([model_a, model_b], base_model=base_model, weights=[0.5, 0.5])
            except Exception:
                pass

            t0 = time.perf_counter()
            try:
                result = mm.merge([model_a, model_b], base_model=base_model, weights=[0.5, 0.5])
                elapsed = time.perf_counter() - t0
                throughput = n_params / elapsed
                label = fmt_ops(n_params)
                row(f"    {label} params", fmt_time(elapsed), f"({fmt_ops(throughput)} params/sec)")
                record(f"Model-{strat_name}", f"{label} params", fmt_ops(throughput), "params/sec")
            except Exception as e:
                label = fmt_ops(n_params)
                row(f"    {label} params", f"ERROR: {e}")
        divider()

    # Clean up GPU memory
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("\n✅ ModelCRDT GPU benchmarks complete")

## 📊 Tabular Merge Benchmarks

DataFrame merge at various scales. Compares Polars engine vs Pandas fallback.

In [ ]:
# SPDX-License-Identifier: BUSL-1.1
# Copyright 2026 Ryan Gillespie / Optitransfer
# Licensed under the Business Source License 1.1
# https://github.com/mgillr/crdt-merge/blob/main/LICENSE
# Change Date: 2028-03-29 | Change License: Apache-2.0

from crdt_merge.dataframe import merge as df_merge
from crdt_merge import HAS_POLARS

print("=" * WIDTH)
print("  Tabular Merge Benchmarks")
print(f"  Polars available: {HAS_POLARS}")
print("=" * WIDTH)

def make_df(n, prefix):
    return [
        {"id": i, "ts": i + (1 if prefix == "b" else 0),
         "name": f"{prefix}_{i}", "score": random.random()}
        for i in range(n)
    ]

for n in [1_000, 10_000, 100_000, 500_000, 1_000_000]:
    df_a = make_df(n, "a")
    df_b = make_df(n, "b")

    t0 = time.perf_counter()
    result = df_merge(df_a, df_b, key="id", timestamp_col="ts")
    elapsed = time.perf_counter() - t0

    label = fmt_ops(n)
    rows_sec = n / elapsed
    row(f"  DataFrame merge ({label} rows)", fmt_time(elapsed), f"({fmt_ops(rows_sec)} rows/sec)")
    row(f"    output rows", f"{len(result)}")
    record("DataFrame", f"merge {label}", fmt_ops(rows_sec), "rows/sec")
    divider()

print("\n✅ Tabular merge benchmarks complete")

## 🌊 Streaming Merge Scale Test

Streaming merge at 1M and 10M rows with memory profiling.

In [ ]:
# SPDX-License-Identifier: BUSL-1.1
# Copyright 2026 Ryan Gillespie / Optitransfer
# Licensed under the Business Source License 1.1
# https://github.com/mgillr/crdt-merge/blob/main/LICENSE
# Change Date: 2028-03-29 | Change License: Apache-2.0

import resource
from crdt_merge.streaming import merge_stream, StreamStats

print("=" * WIDTH)
print("  Streaming Merge Scale Test")
print("=" * WIDTH)

def gen_stream(prefix, n):
    for i in range(n):
        yield {"id": i, "ts": i, "val": f"{prefix}_{i}"}

for n in [1_000_000, 10_000_000]:
    # Memory before
    mem_before = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss

    stats = StreamStats()
    t0 = time.perf_counter()
    total_rows = 0
    for batch in merge_stream(
        gen_stream("a", n), gen_stream("b", n),
        key="id", batch_size=10_000, timestamp_col="ts", stats=stats,
    ):
        total_rows += len(batch)
    elapsed = time.perf_counter() - t0

    mem_after = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    peak_mb = mem_after / 1024  # ru_maxrss is in KB on Linux

    label = fmt_ops(n)
    rows_sec = total_rows / elapsed
    row(f"  Stream merge {label} rows × 2", fmt_time(elapsed))
    row(f"    output rows", f"{total_rows}")
    row(f"    throughput", fmt_ops(rows_sec), "rows/sec")
    row(f"    peak RSS", f"{peak_mb:.0f} MB")
    record("Streaming", f"{label} rows", fmt_ops(rows_sec), "rows/sec")
    divider()

print("\n✅ Streaming merge benchmarks complete")

## 🧪 Full Test Suite

Clone the repo and run the full pytest suite to verify correctness.

In [ ]:
!cd /content && git clone https://github.com/mgillr/crdt-merge.git 2>/dev/null || true
!cd /content/crdt-merge && pip install -q -e ".[all,dev]" pytest-asyncio torch
!cd /content/crdt-merge && python -m pytest tests/ -q --tb=short

## 📋 Results Summary

Consolidated results from all benchmark runs.

In [ ]:
# SPDX-License-Identifier: BUSL-1.1
# Copyright 2026 Ryan Gillespie / Optitransfer
# Licensed under the Business Source License 1.1
# https://github.com/mgillr/crdt-merge/blob/main/LICENSE
# Change Date: 2028-03-29 | Change License: Apache-2.0

print("=" * WIDTH)
print("  crdt-merge v0.8.2 — A100 Benchmark Results Summary")
print("=" * WIDTH)
print()

# Print consolidated table
current_cat = ""
for r in ALL_RESULTS:
    if r["category"] != current_cat:
        current_cat = r["category"]
        print(f"\n  ┌─ {current_cat}")
    unit_str = f" {r['unit']}" if r["unit"] else ""
    print(f"  │  {r['test']:<42s} {r['value']:>20s}{unit_str}")
print(f"  └─")

print()
print("  v0.8.1 → v0.8.2 Notable Changes:")
print("  ─────────────────────────────────")
print("  • NEW: ContextBloom sharded Bloom filter (~10M checks/sec target)")
print("  • NEW: MemorySidecar with sub-µs filter matching")
print("  • NEW: ContextMerge for multi-agent memory deduplication")
print("  • NEW: ContextConsolidator for memory → block compaction")
print("  • IMPROVED: AgentState.merge with confidence-based resolution")
print("  • IMPROVED: SharedKnowledge scaling to 20+ agents")
print()
print("✅ All benchmarks complete")